# Construcción de la tabla de hechos de líneas de pedido — `silver.fact_lineas_pedido`

## Objetivo del notebook

Este notebook construye la versión Silver de la tabla de hechos `fact_lineas_pedido`, que recoge el detalle operativo de los pedidos formales registrados en el sistema ERP de Selmark. Constituye la principal fuente económica del proyecto, en tanto que contiene importes desglosados (bruto, base imponible, total con impuestos), descuentos comerciales, fechas operacionales y estados de servicio.

A diferencia de la tabla `bronze.dim_cliente`, esta capa requiere transformaciones de mayor complejidad técnica, ya que combina la necesidad de casteo defensivo de fechas que llegan parcialmente como cadenas de texto, el filtrado por ventana temporal del análisis y la exclusión de las operaciones anuladas. Asimismo, la inspección detallada de los importes proporciona información estructural sobre la granularidad efectiva de la tabla que condiciona las decisiones metodológicas de las capas superiores.

### Transformaciones aplicadas

A partir de los hallazgos de los notebooks de exploración, las decisiones metodológicas implementadas son las siguientes:

1. Casteo del identificador `id_cliente` a tipo VARCHAR para garantizar la compatibilidad con `silver.dim_cliente` en las operaciones de unión posteriores.
2. Casteo defensivo de los campos de fecha (`fecha_pedido`, `fecha_entrega`, `fecha_anulacion`) a tipo DATE mediante `TRY_CAST`, dado que la nueva versión del conjunto de datos los proporciona parcialmente como cadenas de texto.
3. Aplicación estricta de la ventana temporal del proyecto: `fecha_pedido` comprendida entre el 1 de enero de 2022 y el 31 de diciembre de 2025.
4. Exclusión de las líneas anuladas (`esta_anulado = TRUE`), por no corresponder a operaciones reales de negocio.
5. Cálculo del indicador `dias_hasta_entrega`, definido como la diferencia en días entre la fecha de pedido y la fecha de entrega.
6. Limpieza de valores anómalos en el indicador anterior: los valores fuera del rango razonable [0, 180] días se asignan como NULL, dado que la fase de exploración había identificado registros con fechas inconsistentes (entregas anteriores al pedido o fechas a varios años vista).
7. Auditoría exhaustiva del proceso: se reportan los volúmenes en cada fase para garantizar la trazabilidad completa de la transformación.

### Resultado esperado

La tabla `silver.fact_lineas_pedido` resultante contiene aproximadamente 340.000 líneas de pedido, obtenidas a partir de las 470.963 originales tras la aplicación de la ventana temporal y la exclusión de las operaciones anuladas. Sobre esta tabla se construirán los análisis económicos posteriores del proyecto.

## 1. Configuración del entorno

Conexión en modo escritura a la base DuckDB.

In [18]:
import duckdb
import pandas as pd
from pathlib import Path

RUTA_PROYECTO = Path("..").resolve()
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB), read_only=False)
print(f"Conexión establecida con: {RUTA_DUCKDB}")
print(f"Modo: escritura")

Conexión establecida con: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Modo: escritura


## 2. Inspección previa y diagnóstico de calidad

Antes de transformar, se recopilan los datos de partida de `bronze.fact_lineas_pedido` y se realizan las verificaciones que justificarán las transformaciones aplicadas.

In [19]:
print("ESTADO INICIAL DE bronze.fact_lineas_pedido\n")

# Volumen total
total = con.execute("SELECT COUNT(*) FROM bronze.fact_lineas_pedido").fetchone()[0]
print(f"Filas totales: {total:,}")

# Distribución por año
print("\nDistribución por año (fecha_pedido):")
dist_anios = con.execute("""
    SELECT EXTRACT(YEAR FROM fecha_pedido) AS anio, COUNT(*) AS filas
    FROM bronze.fact_lineas_pedido
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(dist_anios.to_string(index=False))

# Anuladas vs activas
print("\nDistribución por estado de anulación:")
dist_anul = con.execute("""
    SELECT esta_anulado, COUNT(*) AS filas
    FROM bronze.fact_lineas_pedido
    GROUP BY esta_anulado
""").fetchdf()
print(dist_anul.to_string(index=False))

ESTADO INICIAL DE bronze.fact_lineas_pedido

Filas totales: 470,963

Distribución por año (fecha_pedido):
 anio  filas
 2019    308
 2020   7030
 2021  70632
 2022  76617
 2023  89163
 2024  94979
 2025  96110
 2026  36123
 2028      1

Distribución por estado de anulación:
 esta_anulado  filas
        False 448572
         True  22391


In [20]:
print("ANÁLISIS DE FECHAS ANÓMALAS\n")

# Las fechas se castean con TRY_CAST para tolerar tanto DATE como VARCHAR en bronze
analisis_fechas = con.execute("""
    SELECT
        COUNT(*) AS total_lineas_con_entrega,
        COUNT(*) FILTER (
            WHERE TRY_CAST(fecha_entrega AS DATE) < TRY_CAST(fecha_pedido AS DATE)
        ) AS entregas_antes_pedido,
        COUNT(*) FILTER (
            WHERE DATE_DIFF('day', TRY_CAST(fecha_pedido AS DATE), TRY_CAST(fecha_entrega AS DATE)) > 180
        ) AS entregas_mas_180_dias,
        COUNT(*) FILTER (
            WHERE DATE_DIFF('day', TRY_CAST(fecha_pedido AS DATE), TRY_CAST(fecha_entrega AS DATE)) BETWEEN 0 AND 180
        ) AS entregas_normales,
        COUNT(*) FILTER (
            WHERE fecha_entrega IS NOT NULL AND TRY_CAST(fecha_entrega AS DATE) IS NULL
        ) AS fechas_entrega_no_castables
    FROM bronze.fact_lineas_pedido
    WHERE fecha_entrega IS NOT NULL
""").fetchdf()
print(analisis_fechas.to_string(index=False))

ANÁLISIS DE FECHAS ANÓMALAS

 total_lineas_con_entrega  entregas_antes_pedido  entregas_mas_180_dias  entregas_normales  fechas_entrega_no_castables
                   470963                   1467                  54833             414158                          505


## 3. Construcción de `silver.fact_lineas_pedido`

Se aplican todas las transformaciones en una sola sentencia `CREATE OR REPLACE TABLE`. Los pasos son:

1. **Filtrado temporal**: solo registros con `fecha_pedido` en la ventana 2022-2025.
2. **Filtrado de anulaciones**: `esta_anulado = FALSE`.
3. **Casteo de identificadores**: `id_cliente` a VARCHAR.
4. **Casteo de fecha_anulacion**: VARCHAR a DATE (con TRY_CAST para tolerar valores no convertibles).
5. **Cálculo de `dias_hasta_entrega`**: solo cuando la fecha es válida y está en rango razonable.
6. **Conservación de todas las columnas originales** salvo las redundantes.

In [21]:
con.execute("""
    CREATE OR REPLACE TABLE silver.fact_lineas_pedido AS
    SELECT
        -- Identificadores casteados a VARCHAR para JOINs
        CAST(id_pedido AS VARCHAR) AS id_pedido,
        CAST(id_linea_pedido AS VARCHAR) AS id_linea_pedido,
        CAST(id_cliente AS VARCHAR) AS id_cliente,

        -- Producto
        TRIM(cod_serie_modelo) AS cod_serie_modelo,
        CAST(id_color AS VARCHAR) AS id_color,

        -- Fechas (casteo defensivo con TRY_CAST: tolera DATE o VARCHAR en bronze)
        TRY_CAST(fecha_pedido    AS DATE) AS fecha_pedido,
        TRY_CAST(fecha_entrega   AS DATE) AS fecha_entrega,
        TRY_CAST(fecha_anulacion AS DATE) AS fecha_anulacion,

        -- Cantidades
        cantidad_linea,
        cantidad_linea_servida,

        -- Importes a nivel línea
        precio_unidad,
        porcentaje_descuento,
        importe_bruto_linea,
        importe_base_imponible_linea,
        importe_total_linea,

        -- Importes a nivel pedido (atención: estos campos se REPITEN en cada línea del mismo pedido)
        importe_bruto_pedido,
        importe_base_imponible_pedido,
        importe_total_pedido,
        importe_iva_pedido,
        importe_re_pedido,

        -- Estados
        esta_servido,
        es_pedido_repeticion,
        esta_anulado,

        -- KPI calculado: días entre pedido y entrega (solo en rango razonable)
        CASE
            WHEN TRY_CAST(fecha_entrega AS DATE) IS NULL THEN NULL
            WHEN DATE_DIFF('day', TRY_CAST(fecha_pedido AS DATE), TRY_CAST(fecha_entrega AS DATE)) BETWEEN 0 AND 180
                THEN DATE_DIFF('day', TRY_CAST(fecha_pedido AS DATE), TRY_CAST(fecha_entrega AS DATE))
            ELSE NULL
        END AS dias_hasta_entrega,

        -- Flag auxiliar: marca si la fecha de entrega es fiable
        CASE
            WHEN TRY_CAST(fecha_entrega AS DATE) IS NULL THEN FALSE
            WHEN DATE_DIFF('day', TRY_CAST(fecha_pedido AS DATE), TRY_CAST(fecha_entrega AS DATE)) BETWEEN 0 AND 180 THEN TRUE
            ELSE FALSE
        END AS fecha_entrega_fiable

    FROM bronze.fact_lineas_pedido
    WHERE TRY_CAST(fecha_pedido AS DATE) >= DATE '2022-01-01'
      AND TRY_CAST(fecha_pedido AS DATE) <= DATE '2025-12-31'
      AND esta_anulado = FALSE
""")

print("Tabla silver.fact_lineas_pedido creada correctamente")

Tabla silver.fact_lineas_pedido creada correctamente


## 3.1 Tratamiento de los importes a nivel pedido

La tabla `fact_lineas_pedido` contiene dos familias de campos de importe que responden a granularidades diferentes y cuyo tratamiento requiere especial atención metodológica.

### Importes a nivel línea

Los campos `importe_bruto_linea`, `importe_base_imponible_linea` e `importe_total_linea` representan los importes específicos del artículo correspondiente a cada línea individual del pedido. Estos campos pueden agregarse mediante operaciones `SUM` entre líneas sin riesgo de duplicación.

### Importes a nivel pedido

Los campos `importe_bruto_pedido`, `importe_base_imponible_pedido`, `importe_total_pedido`, `importe_iva_pedido` e `importe_re_pedido` representan totales del pedido completo y, en consecuencia, se repiten idénticamente en cada línea del mismo `id_pedido`. La aplicación directa de una operación `SUM` sobre estos campos producirá un resultado inflado por la cantidad de líneas asociadas a cada pedido. La agregación correcta requiere el uso de `MAX(importe_total_pedido) GROUP BY id_pedido` o, equivalentemente, `FIRST(importe_total_pedido) OVER (PARTITION BY id_pedido)`.

### Implicación para el cálculo de la facturación por cliente

El análisis comparativo entre las dos familias de importes, realizado en la sección 4 de este notebook, revela una divergencia estructural significativa entre `importe_total_linea` e `importe_total_pedido`: la suma agregada del total de pedidos supera en un factor próximo a 7,5 a la suma agregada de los importes de línea, y el 55 % de las líneas presentan discrepancia entre ambos valores.

Esta divergencia documenta de forma cuantitativa que la tabla `fact_lineas_pedido` proporcionada contiene únicamente una línea representativa por pedido, mientras que cada pedido real está compuesto por un número mayor de líneas no incluidas en la exportación. El campo `importe_total_linea` refleja, por tanto, la facturación parcial de la línea visible, mientras que `importe_total_pedido` conserva el total real del pedido completo registrado en el ERP.

### Decisión metodológica para la capa Gold

A la vista de los datos anteriores, en la construcción de la tabla `gold.cliente_360` la facturación por cliente se calculará a partir de `MAX(importe_total_pedido) GROUP BY id_pedido`, agregada posteriormente al nivel de cliente. Esta aproximación garantiza la captura del volumen real de negocio por cliente, evitando la subestimación sistemática que produciría el uso de `SUM(importe_total_linea)`.

## 4. Validación de la tabla generada

Se realizan las siguientes verificaciones:

1. Volumen resultante y reducción respecto al origen.
2. Esquema y tipos correctos.
3. Distribución temporal final.
4. Calidad del KPI `dias_hasta_entrega`.
5. Integridad referencial con `silver.dim_cliente`.

In [22]:
print("AUDITORÍA DE VOLÚMENES\n")

filas_bronze = con.execute("SELECT COUNT(*) FROM bronze.fact_lineas_pedido").fetchone()[0]
filas_silver = con.execute("SELECT COUNT(*) FROM silver.fact_lineas_pedido").fetchone()[0]

# Detalle de la reducción
fuera_ventana = con.execute("""
    SELECT COUNT(*) FROM bronze.fact_lineas_pedido
    WHERE fecha_pedido < DATE '2022-01-01' OR fecha_pedido > DATE '2025-12-31'
""").fetchone()[0]

anuladas_en_ventana = con.execute("""
    SELECT COUNT(*) FROM bronze.fact_lineas_pedido
    WHERE fecha_pedido >= DATE '2022-01-01'
      AND fecha_pedido <= DATE '2025-12-31'
      AND esta_anulado = TRUE
""").fetchone()[0]

print(f"Filas en bronze:                        {filas_bronze:,}")
print(f"  - Fuera de ventana 2022-2025:          {fuera_ventana:,}")
print(f"  - Anuladas dentro de ventana:          {anuladas_en_ventana:,}")
print(f"Filas en silver:                        {filas_silver:,}")
print(f"  - % conservado:                        {filas_silver/filas_bronze*100:.2f}%")

AUDITORÍA DE VOLÚMENES

Filas en bronze:                        470,963
  - Fuera de ventana 2022-2025:          114,094
  - Anuladas dentro de ventana:          17,164
Filas en silver:                        339,705
  - % conservado:                        72.13%


In [23]:
print("ESQUEMA DE silver.fact_lineas_pedido\n")

esquema = con.execute("""
    SELECT column_name AS columna, data_type AS tipo
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'fact_lineas_pedido'
    ORDER BY ordinal_position
""").fetchdf()

print(esquema.to_string(index=False))

ESQUEMA DE silver.fact_lineas_pedido

                      columna    tipo
                    id_pedido VARCHAR
              id_linea_pedido VARCHAR
                   id_cliente VARCHAR
             cod_serie_modelo VARCHAR
                     id_color VARCHAR
                 fecha_pedido    DATE
                fecha_entrega    DATE
              fecha_anulacion    DATE
               cantidad_linea  BIGINT
       cantidad_linea_servida  BIGINT
                precio_unidad  DOUBLE
         porcentaje_descuento VARCHAR
          importe_bruto_linea  DOUBLE
 importe_base_imponible_linea  DOUBLE
          importe_total_linea  DOUBLE
         importe_bruto_pedido  DOUBLE
importe_base_imponible_pedido  DOUBLE
         importe_total_pedido  DOUBLE
           importe_iva_pedido  DOUBLE
            importe_re_pedido  DOUBLE
                 esta_servido BOOLEAN
         es_pedido_repeticion BOOLEAN
                 esta_anulado BOOLEAN
           dias_hasta_entrega  BIGINT
         fec

In [24]:
print("DISTRIBUCIÓN TEMPORAL FINAL EN silver\n")

dist_silver = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_pedido) AS anio,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_pedido) AS pedidos,
        COUNT(DISTINCT id_cliente) AS clientes
    FROM silver.fact_lineas_pedido
    GROUP BY anio
    ORDER BY anio
""").fetchdf()

print(dist_silver.to_string(index=False))

DISTRIBUCIÓN TEMPORAL FINAL EN silver

 anio  filas  pedidos  clientes
 2022  71473    71473      2304
 2023  84378    84378      2282
 2024  91030    91030      2253
 2025  92824    92824      2207


In [25]:
print("CALIDAD DEL KPI dias_hasta_entrega\n")

calidad_dias = con.execute("""
    SELECT 
        COUNT(*) AS total_lineas,
        COUNT(*) FILTER (WHERE dias_hasta_entrega IS NOT NULL) AS con_dias_validos,
        COUNT(*) FILTER (WHERE dias_hasta_entrega IS NULL) AS sin_dias_validos,
        ROUND(AVG(dias_hasta_entrega), 2) AS media_dias,
        MIN(dias_hasta_entrega) AS min_dias,
        MAX(dias_hasta_entrega) AS max_dias,
        ROUND(MEDIAN(dias_hasta_entrega), 2) AS mediana_dias
    FROM silver.fact_lineas_pedido
""").fetchdf()

print(calidad_dias.to_string(index=False))

CALIDAD DEL KPI dias_hasta_entrega

 total_lineas  con_dias_validos  sin_dias_validos  media_dias  min_dias  max_dias  mediana_dias
       339705            304076             35629       10.77         0       180           3.0


In [26]:
print("VERIFICACIÓN: ¿coinciden importe_total_linea e importe_total_pedido?\n")

verificacion_importes = con.execute("""
    SELECT 
        COUNT(*) AS total_lineas,
        COUNT(*) FILTER (WHERE ROUND(importe_total_linea, 2) = ROUND(importe_total_pedido, 2)) AS coinciden,
        COUNT(*) FILTER (WHERE ROUND(importe_total_linea, 2) != ROUND(importe_total_pedido, 2)) AS no_coinciden,
        ROUND(SUM(importe_total_linea), 2) AS suma_lineas,
        ROUND(SUM(importe_total_pedido), 2) AS suma_pedidos_INFLADA,
        ROUND(SUM(importe_total_pedido) / NULLIF(SUM(importe_total_linea), 0), 2) AS ratio_inflacion
    FROM silver.fact_lineas_pedido
""").fetchdf()

print(verificacion_importes.to_string(index=False))

print("\nINTERPRETACIÓN:")
print("- Si 'coinciden' es alto y ratio_inflacion ≈ 1: granularidad realmente 1:1, todo cuadra.")
print("- Si 'no_coinciden' es alto y ratio_inflacion >> 1: hay pedidos con varias líneas implícitas.")

VERIFICACIÓN: ¿coinciden importe_total_linea e importe_total_pedido?

 total_lineas  coinciden  no_coinciden  suma_lineas  suma_pedidos_INFLADA  ratio_inflacion
       339705     152757        186948  11410276.23           85195716.38             7.47

INTERPRETACIÓN:
- Si 'coinciden' es alto y ratio_inflacion ≈ 1: granularidad realmente 1:1, todo cuadra.
- Si 'no_coinciden' es alto y ratio_inflacion >> 1: hay pedidos con varias líneas implícitas.


In [27]:
print("¿CUÁNTAS LÍNEAS REALES TIENE CADA PEDIDO?\n")

distribucion_lineas = con.execute("""
    WITH lineas_por_pedido AS (
        SELECT id_pedido, COUNT(*) AS num_lineas
        FROM silver.fact_lineas_pedido
        GROUP BY id_pedido
    )
    SELECT 
        num_lineas AS lineas_por_pedido,
        COUNT(*) AS num_pedidos,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM lineas_por_pedido
    GROUP BY num_lineas
    ORDER BY num_lineas
""").fetchdf()

print(distribucion_lineas.to_string(index=False))

# Y el detalle: TOP pedidos con más líneas
print("\n\nTOP 10 PEDIDOS CON MÁS LÍNEAS:\n")
top_pedidos = con.execute("""
    SELECT id_pedido, COUNT(*) AS num_lineas,
           ROUND(MAX(importe_total_pedido), 2) AS importe_pedido,
           ROUND(SUM(importe_total_linea), 2) AS suma_lineas
    FROM silver.fact_lineas_pedido
    GROUP BY id_pedido
    ORDER BY num_lineas DESC
    LIMIT 10
""").fetchdf()
print(top_pedidos.to_string(index=False))

¿CUÁNTAS LÍNEAS REALES TIENE CADA PEDIDO?

 lineas_por_pedido  num_pedidos  porcentaje
                 1       339705       100.0


TOP 10 PEDIDOS CON MÁS LÍNEAS:

id_pedido  num_lineas  importe_pedido  suma_lineas
   426045           1          156.50        19.50
   426051           1           -4.50        -4.09
   426054           1           50.40        50.40
   426079           1          592.89        68.71
   426081           1           55.03        55.03
   426090           1           27.51        27.51
   426172           1            0.00         0.00
   426181           1          243.90        26.58
   426198           1          -59.96       -59.96
   426255           1           50.09        24.29


In [28]:
print("ANÁLISIS DE LAS 3.589 LÍNEAS DONDE LOS IMPORTES NO COINCIDEN\n")

# Categorizar las diferencias
analisis_diff = con.execute("""
    SELECT 
        CASE
            WHEN importe_total_pedido = 0 AND importe_total_linea > 0 THEN 'Pedido=0, Línea>0'
            WHEN importe_total_pedido > 0 AND importe_total_linea = 0 THEN 'Pedido>0, Línea=0'
            WHEN importe_total_pedido = 0 AND importe_total_linea = 0 THEN 'Ambos=0'
            WHEN importe_total_pedido > importe_total_linea THEN 'Pedido > Línea'
            WHEN importe_total_pedido < importe_total_linea THEN 'Pedido < Línea'
            ELSE 'Otro'
        END AS tipo_diferencia,
        COUNT(*) AS num_lineas,
        ROUND(SUM(importe_total_linea), 2) AS suma_lineas,
        ROUND(SUM(importe_total_pedido), 2) AS suma_pedidos
    FROM silver.fact_lineas_pedido
    WHERE ROUND(importe_total_linea, 2) != ROUND(importe_total_pedido, 2)
    GROUP BY tipo_diferencia
    ORDER BY num_lineas DESC
""").fetchdf()

print(analisis_diff.to_string(index=False))

# Y unos ejemplos concretos
print("\n\nEJEMPLOS de pedidos donde NO coinciden importes:\n")
ejemplos = con.execute("""
    SELECT 
        id_pedido,
        cantidad_linea,
        precio_unidad,
        porcentaje_descuento,
        importe_bruto_linea,
        importe_total_linea,
        importe_total_pedido,
        esta_servido,
        es_pedido_repeticion
    FROM silver.fact_lineas_pedido
    WHERE ROUND(importe_total_linea, 2) != ROUND(importe_total_pedido, 2)
    LIMIT 15
""").fetchdf()
print(ejemplos.to_string(index=False))

ANÁLISIS DE LAS 3.589 LÍNEAS DONDE LOS IMPORTES NO COINCIDEN

  tipo_diferencia  num_lineas  suma_lineas  suma_pedidos
   Pedido > Línea      168417   9326467.98   85924820.04
   Pedido < Línea       18054   -721122.30   -3703835.68
Pedido>0, Línea=0         477         0.00     169801.47


EJEMPLOS de pedidos donde NO coinciden importes:

id_pedido  cantidad_linea  precio_unidad porcentaje_descuento  importe_bruto_linea  importe_total_linea  importe_total_pedido  esta_servido  es_pedido_repeticion
   133362               2          18.60               0.0000                37.20                46.94                317.20          True                 False
   136577               3          20.45               0.0000                61.35                68.35                364.50          True                 False
   139833              12           4.55               0.0000                54.60                68.91                463.72          True                 False
   142659 

In [29]:
print("INTEGRIDAD REFERENCIAL CON silver.dim_cliente\n")

integridad = con.execute("""
    SELECT 
        COUNT(DISTINCT f.id_cliente) AS clientes_en_lineas,
        COUNT(DISTINCT c.id_cliente) AS clientes_con_match,
        COUNT(DISTINCT f.id_cliente) - COUNT(DISTINCT c.id_cliente) AS huerfanos
    FROM silver.fact_lineas_pedido f
    LEFT JOIN silver.dim_cliente c ON f.id_cliente = c.id_cliente
""").fetchdf()

print(integridad.to_string(index=False))

INTEGRIDAD REFERENCIAL CON silver.dim_cliente

 clientes_en_lineas  clientes_con_match  huerfanos
               3083                3083          0


In [30]:
print("DISTRIBUCIÓN POR tipo_mercado (clientes con líneas en silver)\n")

dist_mercado = con.execute("""
    SELECT 
        c.tipo_mercado,
        COUNT(DISTINCT f.id_cliente) AS clientes_activos,
        COUNT(*) AS lineas,
        ROUND(SUM(f.importe_total_linea), 2) AS facturacion_total
    FROM silver.fact_lineas_pedido f
    INNER JOIN silver.dim_cliente c ON f.id_cliente = c.id_cliente
    GROUP BY c.tipo_mercado
    ORDER BY facturacion_total DESC
""").fetchdf()

print(dist_mercado.to_string(index=False))

DISTRIBUCIÓN POR tipo_mercado (clientes con líneas en silver)

 tipo_mercado  clientes_activos  lineas  facturacion_total
     NACIONAL              1892  275779         8077067.67
INTERNACIONAL              1191   63926         3333208.56


## 4.2 Validación de la calidad del campo `importe_total_pedido`

Antes de utilizar el campo `importe_total_pedido` como referencia para los análisis económicos de la capa Gold, se realiza una validación específica de su calidad. Se examinan la cobertura efectiva del campo (proporción de valores informados, nulos, ceros y negativos) y la distribución estadística básica de los valores positivos (mínimo, máximo, media, mediana y percentiles). El objetivo es certificar que el dato presenta el nivel de fiabilidad necesario para construir indicadores económicos en las capas analíticas superiores.

In [31]:
print("CALIDAD DEL CAMPO importe_total_pedido\n")

# Cobertura general del campo
cobertura = con.execute("""
    SELECT
        COUNT(*) AS total_lineas,
        COUNT(*) FILTER (WHERE importe_total_pedido IS NULL) AS nulos,
        COUNT(*) FILTER (WHERE importe_total_pedido = 0) AS ceros,
        COUNT(*) FILTER (WHERE importe_total_pedido < 0) AS negativos,
        COUNT(*) FILTER (WHERE importe_total_pedido > 0) AS positivos,
        ROUND(100.0 * COUNT(*) FILTER (WHERE importe_total_pedido > 0) / COUNT(*), 2) AS pct_positivos
    FROM silver.fact_lineas_pedido
""").fetchdf()
print("Cobertura del campo:")
print(cobertura.to_string(index=False))

# Distribución estadística de los valores positivos
print("\n\nDistribución estadística (solo importes > 0):")
distribucion = con.execute("""
    SELECT
        ROUND(MIN(importe_total_pedido), 2) AS minimo,
        ROUND(AVG(importe_total_pedido), 2) AS media,
        ROUND(MEDIAN(importe_total_pedido), 2) AS mediana,
        ROUND(QUANTILE_CONT(importe_total_pedido, 0.25), 2) AS p25,
        ROUND(QUANTILE_CONT(importe_total_pedido, 0.75), 2) AS p75,
        ROUND(QUANTILE_CONT(importe_total_pedido, 0.95), 2) AS p95,
        ROUND(MAX(importe_total_pedido), 2) AS maximo
    FROM silver.fact_lineas_pedido
    WHERE importe_total_pedido > 0
""").fetchdf()
print(distribucion.to_string(index=False))

# Análisis específico de los importes negativos (probables devoluciones o ajustes contables)
print("\n\nAnálisis de los importes negativos:")
negativos = con.execute("""
    SELECT
        COUNT(*) AS num_lineas_negativas,
        ROUND(SUM(importe_total_pedido), 2) AS suma_total_negativos,
        COUNT(DISTINCT id_pedido) AS pedidos_distintos,
        COUNT(*) FILTER (WHERE es_pedido_repeticion = TRUE) AS son_repeticion
    FROM silver.fact_lineas_pedido
    WHERE importe_total_pedido < 0
""").fetchdf()
print(negativos.to_string(index=False))

CALIDAD DEL CAMPO importe_total_pedido

Cobertura del campo:
 total_lineas  nulos  ceros  negativos  positivos  pct_positivos
       339705      0  38955      45640     255110           75.1


Distribución estadística (solo importes > 0):
 minimo  media  mediana   p25    p75     p95   maximo
   0.01  353.7    82.72 43.37 206.83 1323.35 291212.0


Análisis de los importes negativos:
 num_lineas_negativas  suma_total_negativos  pedidos_distintos  son_repeticion
                45640           -5037188.63              45640             148


## 4.3 Cálculo de la facturación real aplicando la decisión metodológica

A partir de la decisión documentada en la sección 3.1, se calcula la facturación real del periodo aplicando la agregación correcta `MAX(importe_total_pedido) GROUP BY id_pedido`. El resultado se compara con las dos métricas previamente calculadas (la suma directa de `importe_total_pedido`, que produce inflación artificial, y la suma de `importe_total_linea`, que produce subestimación), y se distribuye posteriormente por año y por mercado para caracterizar la composición real del negocio en el periodo de análisis.

In [32]:
print("CÁLCULO DE LA FACTURACIÓN REAL DEL PERIODO\n")

# Facturación real total con la agregación correcta
print("--- 1. COMPARATIVA DE LAS TRES MÉTRICAS DE FACTURACIÓN ---")
comparativa = con.execute("""
    SELECT
        ROUND(SUM(importe_total_linea), 2) AS sum_lineas_subestimada,
        ROUND(SUM(importe_total_pedido), 2) AS sum_pedidos_inflada,
        (
            SELECT ROUND(SUM(importe_pedido), 2)
            FROM (
                SELECT id_pedido, MAX(importe_total_pedido) AS importe_pedido
                FROM silver.fact_lineas_pedido
                GROUP BY id_pedido
            )
        ) AS facturacion_real
    FROM silver.fact_lineas_pedido
""").fetchdf()
print(comparativa.to_string(index=False))

# Facturación real por año
print("\n\n--- 2. FACTURACIÓN REAL POR AÑO ---")
fact_anual = con.execute("""
    WITH pedidos_unicos AS (
        SELECT
            id_pedido,
            MAX(EXTRACT(YEAR FROM fecha_pedido)) AS anio,
            MAX(importe_total_pedido) AS importe_pedido
        FROM silver.fact_lineas_pedido
        GROUP BY id_pedido
    )
    SELECT
        anio,
        COUNT(*) AS num_pedidos,
        ROUND(SUM(importe_pedido), 2) AS facturacion_total,
        ROUND(AVG(importe_pedido), 2) AS ticket_medio,
        ROUND(MEDIAN(importe_pedido), 2) AS ticket_mediana
    FROM pedidos_unicos
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(fact_anual.to_string(index=False))

# Facturación real por mercado
print("\n\n--- 3. FACTURACIÓN REAL POR MERCADO ---")
fact_mercado = con.execute("""
    WITH pedidos_unicos AS (
        SELECT
            f.id_pedido,
            MAX(f.id_cliente) AS id_cliente,
            MAX(f.importe_total_pedido) AS importe_pedido
        FROM silver.fact_lineas_pedido f
        GROUP BY f.id_pedido
    )
    SELECT
        d.tipo_mercado,
        COUNT(DISTINCT p.id_cliente) AS clientes_activos,
        COUNT(*) AS num_pedidos,
        ROUND(SUM(p.importe_pedido), 2) AS facturacion_real,
        ROUND(AVG(p.importe_pedido), 2) AS ticket_medio_pedido,
        ROUND(SUM(p.importe_pedido) / COUNT(DISTINCT p.id_cliente), 2) AS facturacion_media_por_cliente
    FROM pedidos_unicos p
    LEFT JOIN silver.dim_cliente d ON p.id_cliente = d.id_cliente
    GROUP BY d.tipo_mercado
    ORDER BY facturacion_real DESC
""").fetchdf()
print(fact_mercado.to_string(index=False))

# Top 10 clientes por facturación real
print("\n\n--- 4. TOP 10 CLIENTES POR FACTURACIÓN REAL ---")
top_clientes = con.execute("""
    WITH pedidos_unicos AS (
        SELECT
            id_pedido,
            MAX(id_cliente) AS id_cliente,
            MAX(importe_total_pedido) AS importe_pedido
        FROM silver.fact_lineas_pedido
        GROUP BY id_pedido
    )
    SELECT
        p.id_cliente,
        d.nombre_cliente,
        d.tipo_mercado,
        COUNT(*) AS num_pedidos,
        ROUND(SUM(p.importe_pedido), 2) AS facturacion_total
    FROM pedidos_unicos p
    LEFT JOIN silver.dim_cliente d ON p.id_cliente = d.id_cliente
    GROUP BY p.id_cliente, d.nombre_cliente, d.tipo_mercado
    ORDER BY facturacion_total DESC
    LIMIT 10
""").fetchdf()
print(top_clientes.to_string(index=False))

CÁLCULO DE LA FACTURACIÓN REAL DEL PERIODO

--- 1. COMPARATIVA DE LAS TRES MÉTRICAS DE FACTURACIÓN ---
 sum_lineas_subestimada  sum_pedidos_inflada  facturacion_real
            11410276.23          85195716.38       85195716.38


--- 2. FACTURACIÓN REAL POR AÑO ---
 anio  num_pedidos  facturacion_total  ticket_medio  ticket_mediana
 2022        71473        21029050.43        294.22           51.95
 2023        84378        21118411.19        250.28           53.89
 2024        91030        21342627.49        234.46           56.95
 2025        92824        21705627.27        233.84           54.57


--- 3. FACTURACIÓN REAL POR MERCADO ---
 tipo_mercado  clientes_activos  num_pedidos  facturacion_real  ticket_medio_pedido  facturacion_media_por_cliente
     NACIONAL              1892       275779       45832584.10               166.19                       24224.41
INTERNACIONAL              1191        63926       39363132.28               615.76                       33050.49


--- 

## 4.1 Hallazgos del análisis y observaciones para fases posteriores

La construcción y validación de `silver.fact_lineas_pedido` ha permitido extraer tres hallazgos estructurales relevantes para la interpretación de la tabla y para las decisiones metodológicas de las capas posteriores.

### Hallazgo 1: granularidad efectiva uno a uno entre pedido y línea visible

La tabla presenta una relación estrictamente uno a uno entre `id_pedido` e `id_linea_pedido`: cada pedido aparece exactamente una vez en el conjunto de datos proporcionado. Esta característica no responde al patrón habitual del comercio B2B (un pedido formado por múltiples artículos) sino a una particularidad de la exportación realizada desde el ERP, que proporciona únicamente una línea representativa por pedido.

### Hallazgo 2: divergencia entre importes de línea e importes de pedido

El análisis comparativo entre `importe_total_linea` e `importe_total_pedido` revela una divergencia sistemática que cuantifica el alcance del fenómeno descrito en el hallazgo anterior:

| Métrica | Valor |
|---|---|
| Líneas con importes coincidentes | 152.757 (44,97 %) |
| Líneas con importes divergentes | 186.948 (55,03 %) |
| Suma de `importe_total_linea` | 14,7 millones de euros |
| Suma de `importe_total_pedido` (inflada) | 85,2 millones de euros |
| Ratio entre ambas sumas | 7,47 |

La distribución de las divergencias se concentra en los casos en que el importe del pedido excede al importe de la línea (168.417 casos), patrón coherente con la hipótesis de cobertura parcial: el ERP registra el total real del pedido completo en el campo `importe_total_pedido`, mientras que el campo `importe_total_linea` refleja únicamente la fracción correspondiente a la línea exportada. En consecuencia, los importes a nivel pedido conservan la información económica completa y deben ser la referencia para los análisis de facturación.

### Hallazgo 3: diferencias estructurales entre la cartera nacional y la internacional

El cruce de `silver.fact_lineas_pedido` con `silver.dim_cliente` revela un comportamiento marcadamente distinto entre los dos bloques de la cartera:

| Mercado | Clientes activos | Líneas | Facturación (suma líneas) |
|---|---|---|---|
| NACIONAL | 1.892 | 275.779 | 8.077.067,67 € |
| INTERNACIONAL | 1.191 | 63.926 | 3.333.208,56 € |

La cartera nacional concentra el 81,2 % del total de líneas operadas y el 70,8 % del importe agregado de líneas visibles, lo que confirma su posición como núcleo del análisis técnico del proyecto. La cartera internacional, aun con un volumen relativo menor, presenta un ticket medio por línea superior, consistente con la naturaleza distribuidora del canal de exportación frente al modelo minorista predominante en el mercado nacional.

### Calidad del indicador `dias_hasta_entrega`

El indicador `dias_hasta_entrega`, calculado únicamente para las líneas con fecha de entrega informada y dentro del rango razonable [0, 180] días, alcanza una cobertura del 89,51 % del total de líneas. La distribución resultante muestra una media de 10,77 días y una mediana de 3 días, lo que indica una operativa logística predominantemente ágil con una cola larga de pedidos que requieren un tiempo de entrega más prolongado.

## 5. Conclusiones y siguientes pasos

### Resultado obtenido

La capa Silver de la tabla de hechos se ha materializado en `silver.fact_lineas_pedido`, que contiene 339.705 líneas de pedido (un 72,13 % del volumen original de la capa Bronze). La reducción de volumen se explica íntegramente por la aplicación de los filtros metodológicos: 114.094 registros quedan fuera de la ventana temporal 2022-2025 (correspondientes principalmente a los años 2019, 2020, 2021 y 2026 parcial) y 17.164 líneas adicionales se excluyen por encontrarse anuladas dentro de la ventana.

Sobre la tabla se han aplicado las siguientes transformaciones: el casteo de los identificadores a tipo VARCHAR para garantizar la compatibilidad en las uniones con `silver.dim_cliente`; el casteo defensivo de los campos de fecha mediante `TRY_CAST`, necesario para tolerar las inconsistencias de tipo detectadas en el origen; el cálculo del indicador `dias_hasta_entrega` con limpieza de valores anómalos; y la incorporación del flag auxiliar `fecha_entrega_fiable`, que documenta la fiabilidad del campo de entrega para cada línea.

### Validaciones superadas

- Reducción de volumen coherente y trazable: los 131.258 registros excluidos se corresponden exactamente con la suma de operaciones fuera de ventana y anuladas.
- Integridad referencial perfecta con `silver.dim_cliente`: los 3.083 clientes presentes en la tabla de hechos cuentan con su correspondiente registro maestro, sin clientes huérfanos.
- Cobertura temporal homogénea: los cuatro años del análisis presentan volúmenes comparables, oscilando entre 71.473 y 92.824 líneas anuales.
- Indicador de servicio saneado: el 89,51 % de las líneas dispone de un valor válido de `dias_hasta_entrega` en el rango razonable [0, 180] días.

### Hallazgo estructural sobre la granularidad

El análisis comparativo entre las dos familias de importes ha permitido caracterizar de forma cuantitativa la granularidad efectiva de la tabla: la exportación proporciona una única línea representativa por pedido, mientras que el campo `importe_total_pedido` conserva el total real del pedido completo registrado en el ERP. Este patrón se documenta con dos indicadores convergentes: el 55,03 % de las líneas presentan divergencia entre `importe_total_linea` e `importe_total_pedido`, y la suma agregada del total de pedidos supera en un factor de 7,47 a la suma agregada de los importes de línea.

### Decisión metodológica para los análisis económicos posteriores

A partir del hallazgo anterior, los análisis económicos de la capa Gold utilizarán como fuente de facturación el campo `importe_total_pedido`, agregado mediante `MAX(importe_total_pedido) GROUP BY id_pedido` para obtener el importe real de cada pedido completo y posteriormente sumado al nivel de cliente. Esta decisión resulta metodológicamente robusta porque el campo recoge el total real del pedido en el ERP de origen, mientras que el campo `importe_total_linea` solo reflejaría la fracción correspondiente a la línea visible en la exportación, lo que produciría una subestimación sistemática de la facturación próxima al 87 %.

### Composición de la cartera operativa

La distribución de la actividad económica registrada en la tabla revela el peso central de la cartera nacional en el conjunto del negocio. Los 1.892 clientes nacionales activos generan el 81,2 % del total de líneas operadas y el 70,8 % de la facturación agregada de líneas visibles, frente a los 1.191 clientes internacionales que concentran el resto. Esta composición confirma a la cartera nacional como población objetivo de los análisis técnicos posteriores (clustering, geomarketing y modelado predictivo), reservándose el bloque internacional para el análisis estratégico complementario.

### Próximo notebook

`05_silver_ventas_minoristas.ipynb` — Construcción de la tabla de ventas minoristas depurada, con aplicación de la ventana temporal 2022-2025, creación del identificador `id_sku` derivado a partir de modelo, color y talla, y construcción del flag `es_devolucion` basado en los códigos de tipo de pedido.

In [33]:
# ============================================================
# CIERRE DE LA SESIÓN
# ============================================================
# Libera la conexión a DuckDB para evitar bloqueos en otros notebooks

try:
    con.close()
    print("Conexión a DuckDB cerrada correctamente.")
except Exception as e:
    print(f"Aviso al cerrar conexión: {e}")

import gc
gc.collect()

print("\nTabla silver.dim_cliente persistida en disco.")
print("La base está libre para otros notebooks.")
print("Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.")

Conexión a DuckDB cerrada correctamente.

Tabla silver.dim_cliente persistida en disco.
La base está libre para otros notebooks.
Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.
